# 04. PFN: 1D GP 회귀 재현

세션 1(이론, 노트북 01~03)에서 이해한 내용 — GP prior, prior-fitting NLL, set을 처리하는 Transformer encoder, bar distribution — 을 하나로 엮어 **PFN을 직접 학습**합니다.  
목표는 노트북 03 §8의 **analytic GP posterior(정답)** 를 학습된 PFN이 forward pass 한 번으로 재현 되는 지 확인하는 것입니다.

GP를 재현 대상으로 고른 이유는 posterior가 닫힌형으로 정확히 계산돼, "PFN이 논문만큼 잘 하는가"를 정답과 직접 대조할 수 있기 때문입니다. 하이퍼파라미터는 논문(§5.1 / 부록 F.1)값 **RBF lengthscale 0.6, variance 1.0, 관측 노이즈 1e-4** 로 고정합니다.

In [ ]:
try:
    import pfn_cookbook
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import pfn_cookbook

import os
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch

from pfn_cookbook import (
    PFNConfig,
    analytic_gp_posterior,
    build_criterion,
    build_model,
    build_prior_sampler,
    get_device,
    pfn_predict,
    set_seed,
    train_pfn,
)

set_seed(42)
device = get_device()
if device.type == "mps":
    device = torch.device("cpu")

CONFIG = PFNConfig.small()
EXPERIMENT = "pfn-gp-reproduction"
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    repo_root = Path.cwd().parent
    mlflow.set_tracking_uri(f"sqlite:///{repo_root / 'mlflow.db'}")
    if mlflow.get_experiment_by_name(EXPERIMENT) is None:
        mlflow.create_experiment(EXPERIMENT, artifact_location=(repo_root / "mlruns").as_uri())
    mlflow.set_experiment(EXPERIMENT)

print("device:", device)
print("config:", CONFIG)

## 1. 정답: analytic GP posterior

노트북 03 §8과 동일한 설정으로 정답을 계산합니다. 
context 관측 `D = (X_tr, y_tr)` 5개가 주어졌을 때의 GP posterior 평균·분산이며, 아래 학습된 PFN의 예측을 이 곡선에 겹쳐 재현 성공을 판정합니다.

In [ ]:
X_grid = np.linspace(-5, 5, 200)
X_tr = np.array([-4.0, -2.5, -0.5, 1.0, 3.5])
set_seed(0)
y_tr = np.sin(X_tr) + 1e-4 * np.random.randn(X_tr.size)  # 생성 노이즈 = assumed noise

mu, cov = analytic_gp_posterior(
    X_tr, y_tr, X_grid
)  # 기본값 ls=0.6, var=1.0, noise=1e-4
sd = np.sqrt(np.clip(np.diag(cov), 0, None))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.fill_between(
    X_grid, mu - 2 * sd, mu + 2 * sd, color="C0", alpha=0.2, label="analytic ±2σ"
)
ax.plot(X_grid, mu, color="C0", lw=2, label="analytic posterior mean")
ax.scatter(X_tr, y_tr, c="black", s=35, zorder=5, label="context D")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Reproduction target: analytic GP posterior (ls=0.6, noise=1e-4)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 2. GP prior 샘플러 — PFN이 학습하는 분포

`GPPriorSampler`는 매 학습 스텝의 데이터셋을 batch로 뽑습니다.  

한 데이터셋 = 입력 `x ~ U(-5, 5)`와 그 x에서 zero-mean GP에서 뽑은 라벨 `y ~ N(0, K(x, x) + noise²I)`

아래 prior 함수 샘플이 PFN 학습 데이터의 원천이며 lengthscale이 0.6으로 짧아 함수가 곡선형입니다.

In [ ]:
prior_sampler = build_prior_sampler(CONFIG)

set_seed(1)
xs, ys = prior_sampler.sample_batch(batch_size=6, seq_len=200, device="cpu")
order = torch.argsort(xs[..., 0], dim=1)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i in range(6):
    xi = xs[i, order[i], 0].numpy()
    yi = ys[i, order[i]].numpy()
    ax.plot(xi, yi, lw=1.0, alpha=0.8)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("GP prior function samples (source of PFN training data)")
plt.show()

## 3. bar distribution — 회귀 출력 표현

PFN은 query마다 bucket 위의 범주분포(softmax)를 내고, bucket 안을 균일 밀도로 보아 계단형 예측분포를 만듭니다.  
학습 기본값은 `FullSupportBarDistribution`으로, 양 끝 bucket을 half-normal tail로 바꿔 `[-4, 4]` 밖의 y에도 유한한 NLL을 줍니다.  
아래는 무작위 logits 하나가 만드는 예측 밀도를 시각화한 것 입니다.

In [ ]:
criterion = build_criterion(CONFIG)

set_seed(2)
demo_logits = torch.randn(CONFIG.num_buckets) * 1.5
grid = torch.linspace(-5, 5, 500)
dens = criterion.pdf(demo_logits, grid).numpy()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(grid.numpy(), dens, color="C0")
ax.fill_between(grid.numpy(), dens, color="C0", alpha=0.25)
ax.set_xlabel("y")
ax.set_ylabel("density")
ax.set_title(f"A predictive density from bar distribution (B={CONFIG.num_buckets})")
plt.show()

print("mean:", criterion.mean(demo_logits).item(), "| std:", criterion.std(demo_logits).item())

## 4. PFN 모델 — set을 처리하는 Transformer encoder

각 (x, y)가 토큰 하나입니다.
- context 토큰은 실제 y 로
- query 토큰은 mask 토큰 사용

positional encoding이 없어 순서에 불변이며, attention mask로 **query는 context에만 attend**(다른 query·자기 타깃은 안 봄)하게 해 한 번에 여러 query를 독립적으로 예측합니다.

출력 head는 query마다 bar distribution logits를 출력합니다.

In [ ]:
model = build_model(CONFIG).to(device)
print("parameters:", f"{model.num_parameters():,}")
print(model.encoder)

## 5. 학습 — prior-fitting 루프

매 스텝 GP prior에서 데이터셋 batch를 뽑아 context/query로 나누고, query 라벨의 NLL을 최소화합니다.(논문 Algorithm 1). 이 때 실제 데이터는 사용되지 않습니다. 아래 loss 곡선이 내려가면 PFN이 "이 prior에서 나온 데이터셋의 PPD"를 학습하고 있는 것입니다.

In [ ]:
set_seed(42)
run_name = f"gp-pfn-emsize{CONFIG.emsize}-steps{CONFIG.steps}"
with mlflow.start_run(run_name=run_name) as run:
    # config는 params로, step별 train NLL·lr은 metric으로 기록된다
    history = train_pfn(
        model, prior_sampler, criterion, CONFIG, device=device, log_every=100, mlflow_log=True
    )

    losses = np.array(history["losses"])
    kernel = np.ones(25) / 25
    smooth = np.convolve(losses, kernel, mode="valid")

    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    ax.plot(losses, color="C0", alpha=0.3, lw=0.8, label="loss")
    ax.plot(np.arange(len(smooth)) + 12, smooth, color="C0", lw=2, label="smoothed")
    ax.set_xlabel("step")
    ax.set_ylabel("query NLL")
    ax.set_title("Prior-fitting loss")
    ax.legend()
    mlflow.log_figure(fig, "plots/loss_curve.png")
    plt.show()

RUN_ID = run.info.run_id  # 아래 검증 지표도 같은 run에 붙인다
print("MLflow run:", RUN_ID)
print(
    "initial ~",
    round(float(losses[:20].mean()), 3),
    "| final ~",
    round(float(losses[-50:].mean()), 3),
)

## 6. 검증 — PFN 예측을 정답 posterior에 겹치기

§1의 정답과 정확히 같은 setup(같은 context D, 같은 grid)을 학습된 PFN에 in-context로 넣어 예측분포를 얻고, analytic GP posterior에 겹칩니다. 두 평균이 겹치고 ±2σ 밴드가 대체로 일치하면 재현 성공입니다.

In [ ]:
pred = pfn_predict(
    model,
    criterion,
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
    torch.tensor(X_grid, dtype=torch.float32),
)
pfn_mean = pred["mean"].cpu().numpy()
pfn_sd = pred["std"].cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.fill_between(X_grid, mu - 2 * sd, mu + 2 * sd, color="C0", alpha=0.15, label="analytic ±2σ")
ax.plot(X_grid, mu, color="C0", lw=2, label="analytic mean")
ax.plot(X_grid, pfn_mean, color="C3", lw=2, ls="--", label="PFN mean")
ax.fill_between(
    X_grid, pfn_mean - 2 * pfn_sd, pfn_mean + 2 * pfn_sd, color="C3", alpha=0.15, label="PFN ±2σ"
)
ax.scatter(X_tr, y_tr, c="black", s=35, zorder=5, label="context D")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("PFN prediction vs analytic GP posterior")
ax.legend(loc="upper left", fontsize=8)

rmse_mean = float(np.sqrt(np.mean((pfn_mean - mu) ** 2)))
rmse_sd = float(np.mean(np.abs(pfn_sd - sd)))
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_figure(fig, "plots/pfn_vs_gp_posterior.png")
    mlflow.log_metrics({"rmse_mean": rmse_mean, "mae_std": rmse_sd})
plt.show()

print(f"RMSE(mean) = {rmse_mean:.4f}  |  mean|Δσ| = {rmse_sd:.4f}")

In [ ]:
# NLL 비교: query 점에서 PFN NLL vs Gaussian analytic-GP NLL (GP가 Bayes-optimal 하한)
Xq = np.linspace(-4.5, 4.5, 40)
yq = np.sin(Xq)

mu_q, cov_q = analytic_gp_posterior(X_tr, y_tr, Xq)
var_q = np.clip(np.diag(cov_q), 1e-9, None) + 1e-4**2
gp_nll = float(np.mean(0.5 * np.log(2 * np.pi * var_q) + 0.5 * (yq - mu_q) ** 2 / var_q))

predq = pfn_predict(
    model,
    criterion,
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
    torch.tensor(Xq, dtype=torch.float32),
)
pfn_nll = float(criterion(predq["logits"], torch.tensor(yq, dtype=torch.float32)).mean())

print(f"GP analytic NLL (하한): {gp_nll:.3f}")
print(f"PFN NLL              : {pfn_nll:.3f}")
print(f"gap                  : {pfn_nll - gp_nll:+.3f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({"gp_nll": gp_nll, "pfn_nll": pfn_nll, "nll_gap": pfn_nll - gp_nll})

## 7. context 0개 → prior 회복

context를 하나도 주지 않으면 PFN은 GP **prior** 를 반환해야 합니다:  
평균 ≈ 0, 표준편차 ≈ √(variance + noise²) ≈ 1.0.

이 값이 나오면 PFN이 특정 데이터를 외운 게 아니라 prior 자체를 학습했다는 증거로 볼 수 있습니다.

In [ ]:
pred0 = pfn_predict(
    model,
    criterion,
    torch.zeros(0, dtype=torch.float32),
    torch.zeros(0, dtype=torch.float32),
    torch.tensor(X_grid, dtype=torch.float32),
)
m0 = pred0["mean"].cpu().numpy()
s0 = pred0["std"].cpu().numpy()
print(f"zero-context  mean ≈ {m0.mean():+.3f} (목표 0)  |  std ≈ {s0.mean():.3f} (목표 ~1.0)")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({"zero_ctx_mean": float(m0.mean()), "zero_ctx_std": float(s0.mean())})